In [1]:
import numpy as np
import pandas as pd
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel, VarianceThreshold, SelectPercentile
from sklearn.model_selection import RandomizedSearchCV, RepeatedStratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from dotenv import load_dotenv
import joblib
from gene_thesaurus import GeneThesaurus


In [11]:
load_dotenv()

X_train_path = os.getenv("X_TRAIN_CSV_PATH")
y_train_path = os.getenv("Y_TRAIN_TSV_PATH")
X_test_path = os.getenv("X_TEST_CSV_PATH")
y_test_path = os.getenv("Y_TEST_TSV_PATH")
model_path = os.getenv("RF_MODEL_PATH")
selected_features_path = os.getenv("SELECTED_FEATURES_PATH")
hyperparameters_path = os.getenv("HYPERPARAMETERS_PATH")
pipeline_path = os.getenv("PIPELINE_PATH")
random_seed = int(os.getenv("RANDOM_SEED"))
n_jobs = int(os.getenv("N_JOBS", -1))

# Print configuration
print(f"X_train_path: {X_train_path}")
print(f"y_train_path: {y_train_path}")
print(f"X_test_path: {X_test_path}")
print(f"y_test_path: {y_test_path}")
print(f"model_path: {model_path}")
print(f"selected_features_path: {selected_features_path}")
print(f"hyperparameters_path: {hyperparameters_path}")
print(f"pipeline_path: {pipeline_path}")
print(f"random_seed: {random_seed}")
print(f"n_jobs: {n_jobs}")

# Hardcoded variables
VARIANCE_THRESHOLD=0.01  # REMOVE NEAR-CONSTANT GENES
FEATURE_SELECTION_PERCENTILE=20  # KEEP TOP n% OF FEATURES
FINAL_N_ESTIMATORS=500

X_train_path: /home/mariya/Data/pallas/nf-core-gexml-output-train/transpose/transposed.csv
y_train_path: /home/mariya/Data/pallas/nf-core-gexml-output-train/class/subtypes.tsv
X_test_path: /home/mariya/Data/pallas/nf-core-gexml-output-cal/transpose/transposed.csv
y_test_path: /home/mariya/Data/pallas/nf-core-gexml-output-cal/class/subtypes.tsv
model_path: /home/mariya/Data/pallas/training/gex/gex.model.pkl
selected_features_path: /home/mariya/Data/pallas/training/gex/gex.selected_features.pkl
hyperparameters_path: /home/mariya/Data/pallas/training/gex/gex.hyperparameters.pkl
pipeline_path: /home/mariya/Data/pallas/training/gex/gex.pipeline.pkl
random_seed: 42
n_jobs: 10


In [3]:
# Read in X_train and y_train
X_train = pd.read_csv(X_train_path, index_col=0)
X_train


,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938,ENSG00000000971,ENSG00000001036,ENSG00000001084,ENSG00000001167,...,ENSG00000293642,ENSG00000293661,ENSG00000293662,ENSG00000293663,ENSG00000293685,ENSG00000293689,ENSG00000293691,ENSG00000300510,ENSG00000310517,ENSG00000310525
id,,,,,,,,,,,,,,,,,,,,,
SJALL015946_D1,-0.529331,-1.282665,4.496392,3.002916,3.679745,3.919713,2.030881,2.653642,5.433819,6.181622,...,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665
SJALL015985_D1,-1.282665,-1.282665,4.426391,3.565546,4.528216,1.576876,1.898853,4.328216,4.068370,7.474183,...,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665
SJALL015988_D1,-1.067062,-1.282665,4.262553,4.118804,4.436401,4.352112,4.482554,1.618192,4.009013,6.716113,...,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665
SJALL050885_D1,1.656285,-1.282665,3.653062,3.655525,4.331860,5.722800,2.907099,2.522142,4.557718,6.518449,...,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665
SJAML001409_D2,-1.282665,-1.282665,3.946285,5.510030,4.019000,3.593475,2.516766,4.811709,4.130755,6.316438,...,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Case_91,1.142084,-1.282665,4.308610,3.822912,4.679321,8.736406,4.766900,6.885720,4.727378,5.200539,...,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665
Case_92,-1.282665,-1.282665,4.084838,3.416277,3.838330,6.680805,1.390596,3.497792,4.297127,5.536738,...,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665
Case_95,0.299984,-1.282665,3.650709,2.127346,1.816580,7.272053,-1.023101,6.525504,6.340253,6.753698,...,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665


In [4]:
y_train = pd.read_csv(y_train_path, sep="\t", index_col=0)
y_train

,class
sample,
TCGA-AB-2875,"AML, RUNX1::RUNX1T1"
TARGET-20-PARXMP,"AML, NUP98-r"
TARGET-20-PANZKA,"B-ALL, KMT2A-r"
TARGET-20-PAVIXV,"AML, NPM1alt"
TARGET-20-PASHBI,"AML, RUNX1::RUNX1T1"
...,...
817_T,Non-leukemic
867_B,Non-leukemic
845_B,Non-leukemic


In [5]:
# align the indices
X_train = X_train.loc[y_train.index]
X_train

,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938,ENSG00000000971,ENSG00000001036,ENSG00000001084,ENSG00000001167,...,ENSG00000293642,ENSG00000293661,ENSG00000293662,ENSG00000293663,ENSG00000293685,ENSG00000293689,ENSG00000293691,ENSG00000300510,ENSG00000310517,ENSG00000310525
sample,,,,,,,,,,,,,,,,,,,,,
TCGA-AB-2875,-1.282665,-1.282665,3.964234,4.154264,5.089640,4.218721,1.863897,4.137050,4.704880,6.910367,...,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665
TARGET-20-PARXMP,-1.282665,-0.502498,4.973305,3.534769,4.800335,4.813971,2.477810,3.830735,7.155263,7.490717,...,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665
TARGET-20-PANZKA,-1.282665,-1.282665,6.124143,3.617044,3.426577,4.956074,1.198352,5.415927,4.344694,5.778934,...,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665
TARGET-20-PAVIXV,-1.282665,-1.282665,4.511213,3.693262,4.464065,2.434690,-0.074680,3.459821,4.824460,6.832382,...,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665
TARGET-20-PASHBI,-1.282665,-1.018019,5.306492,4.008920,4.391952,3.266210,2.929776,4.623567,5.207011,6.076751,...,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
817_T,2.503579,-1.282665,4.238758,4.374576,3.308737,6.434140,3.996499,4.207382,4.555174,5.772698,...,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665
867_B,-0.925908,-1.282665,4.193046,4.471986,3.725706,7.459025,-1.223899,4.692828,4.509541,5.640395,...,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665
845_B,-0.382782,-1.282665,4.044961,4.484361,3.901193,7.357793,0.119261,4.318384,4.645693,5.984236,...,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665,-1.282665


In [6]:
X_test = pd.read_csv(X_test_path, index_col=0)
y_test = pd.read_csv(y_test_path, sep="\t", index_col=0)
y_test

,class
sample,
SJBALL022245,"B-ALL, BCR::ABL1-like"
SJBALL020936,"B-ALL, BCR::ABL1-like"
SJERG020305,"B-ALL, DUX4-r"
SJE2A056,"B-ALL, high hyperdiploid"
SJETV070,"B-ALL, ETV6::RUNX1"
...,...
Case_7,"B-ALL, KMT2A-r"
Case_153,"B-ALL, high hyperdiploid"
ALL_808,"B-ALL, KMT2A-r"


In [7]:
# align the indices
X_test = X_test.loc[y_test.index]
X_test

,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938,ENSG00000000971,ENSG00000001036,ENSG00000001084,ENSG00000001167,...,ENSG00000293642,ENSG00000293661,ENSG00000293662,ENSG00000293663,ENSG00000293685,ENSG00000293689,ENSG00000293691,ENSG00000300510,ENSG00000310517,ENSG00000310525
sample,,,,,,,,,,,,,,,,,,,,,
SJBALL022245,-1.160736,-1.284493,3.690224,3.842920,3.990519,6.910326,4.158515,2.928164,4.371870,6.269796,...,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493
SJBALL020936,-0.916217,-1.284493,3.930558,3.935421,4.320277,5.262583,1.429076,1.276617,3.803745,6.292126,...,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493
SJERG020305,-1.284493,-1.284493,3.887017,3.138024,4.120716,4.655123,0.301186,2.247712,5.095108,7.481786,...,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493
SJE2A056,-0.350102,-1.284493,3.841476,4.560256,4.174419,4.462095,-0.227813,2.227689,4.204744,6.409967,...,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493
SJETV070,-0.993628,-1.284493,3.743854,4.082922,4.244401,5.903465,2.213111,3.658550,5.292338,6.682809,...,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Case_7,-1.284493,-1.284493,3.653207,4.207138,3.641672,4.392838,-1.284493,5.838627,4.200710,5.441259,...,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493
Case_153,1.014555,-0.339060,3.845954,2.553554,4.550329,6.860235,4.561062,5.313530,4.259746,6.631624,...,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493
ALL_808,1.343191,-1.284493,4.273148,2.909243,2.082523,4.361824,0.024988,3.706412,5.340863,6.695113,...,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493,-1.284493


In [8]:
# Get least frequently ocurring class count
min_class_count = y_train.value_counts().min()
min_class_count

np.int64(3)

In [9]:
y_train = y_train.iloc[:, 0].values  # Convert to 1D numpy array
y_test = y_test.iloc[:, 0].values
y_test

array(['B-ALL, BCR::ABL1-like', 'B-ALL, BCR::ABL1-like', 'B-ALL, DUX4-r',
       'B-ALL, high hyperdiploid', 'B-ALL, ETV6::RUNX1', 'B-ALL, KMT2A-r',
       'B-ALL, DUX4-r', 'B-ALL, ETV6::RUNX1', 'B-ALL, ETV6::RUNX1',
       'B-ALL, KMT2A-r', 'B-ALL, DUX4-r', 'B-ALL, MEF2D-r',
       'B-ALL, KMT2A-r', 'B-ALL, BCR::ABL1-like',
       'B-ALL, high hyperdiploid', 'B-ALL, DUX4-r', 'B-ALL, TCF3::PBX1',
       'B-ALL, KMT2A-r', 'B-ALL, TCF3::PBX1', 'B-ALL, KMT2A-r',
       'B-ALL, DUX4-r', 'B-ALL, DUX4-r', 'B-ALL, high hyperdiploid',
       'B-ALL, ETV6::RUNX1-like', 'B-ALL, ETV6::RUNX1', 'B-ALL, PAX5alt',
       'B-ALL, high hyperdiploid', 'B-ALL, BCR::ABL1-like',
       'B-ALL, HLF-r', 'B-ALL, high hyperdiploid', 'B-ALL, hypodiploid',
       'B-ALL, BCR::ABL1-like', 'B-ALL, BCR::ABL1-like', 'B-ALL, KMT2A-r',
       'B-ALL, KMT2A-r', 'B-ALL, MEF2D-r', 'B-ALL, MEF2D-r',
       'B-ALL, high hyperdiploid', 'B-ALL, ETV6::RUNX1-like',
       'B-ALL, BCR::ABL1-like', 'B-ALL, BCR::ABL1-like',
     

In [12]:
N_ITER=30
N_REPEATS=1 # WE WILL VALIDATE FINAL PARAMS USING MORE REPEATS

# ----------------------
# Pipeline: Variance filter -> RF feature selection -> RF classifier
# ----------------------
pipeline = Pipeline([
    ("variance_filter", VarianceThreshold(threshold=VARIANCE_THRESHOLD)), # remove near-constant genes
    ("feature_selection", SelectPercentile(percentile=FEATURE_SELECTION_PERCENTILE)),  # keep only top 10% of features
    ("rf", RandomForestClassifier(random_seed, n_jobs=-n_jobs))
])

# ----------------------
# Hyperparameter search space
# ----------------------
param_grid = {
    'rf__n_estimators': [50, 100, 200],
    'rf__max_depth': [None, 20, 50],
    'rf__max_features': ['sqrt', 'log2'],
    'rf__min_samples_split': [2, 5, 10],
    'rf__min_samples_leaf': [1, 2, 5],
    'rf__class_weight': [None, 'balanced']
}

# ----------------------
# Cross-validation + randomized search
# ----------------------
cv = RepeatedStratifiedKFold(n_splits=min_class_count,
                             n_repeats=N_REPEATS,
                             random_state=random_seed)

search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_grid,
    n_iter=N_ITER,
    cv=cv,
    scoring="f1_weighted",   # good for multiclass imbalance
    n_jobs=-n_jobs,
    random_state=random_seed,
    verbose=1
)

search.fit(X_train, y_train)

print("Best hyperparameters:", search.best_params_)
print("Best CV (F1-weighted):", search.best_score_)
best_pipeline = search.best_estimator_

## SAVE EVERYTHING
# Save best hyperparameters
joblib.dump(search.best_params_, hyperparameters_path)

# Save features
# First get features that passed the variance threshold
variance_mask = best_pipeline.named_steps['variance_filter'].get_support()
remaining_features = X_train.columns[variance_mask]

# Then get features selected by SelectFromModel
selected_mask = best_pipeline.named_steps['feature_selection'].get_support()
selected_features = remaining_features[selected_mask]

joblib.dump(selected_features, selected_features_path)

# Save the whole pipeline
joblib.dump(best_pipeline, pipeline_path)

Fitting 3 folds for each of 30 candidates, totalling 90 fits


Best hyperparameters: {'rf__n_estimators': 200, 'rf__min_samples_split': 10, 'rf__min_samples_leaf': 5, 'rf__max_features': 'sqrt', 'rf__max_depth': 50, 'rf__class_weight': 'balanced'}
Best CV (F1-weighted): 0.9089367188373211


['/home/mariya/Data/pallas/training/gex/gex.pipeline.pkl']

In [13]:
# Cross-validate again with more repeats and larger estimators
N_REPEATS=10

# Use just the selected features
X_train_selected = X_train[selected_features]
X_test_selected = X_test[selected_features]

# Read in "best_hyperparameters.pkl"
with open(hyperparameters_path, "rb") as f:
    best_params = joblib.load(f)

best_params = {k.replace('rf__', ''): v for k, v in best_params.items()}
best_params['n_estimators'] = FINAL_N_ESTIMATORS
print("Final model parameters:", best_params)
final_rf = RandomForestClassifier(**best_params,
                            n_jobs=n_jobs,
                            random_state=random_seed)

# Cross-validate again with more repeats
cv = RepeatedStratifiedKFold(n_splits=min_class_count,
                             n_repeats=N_REPEATS,
                             random_state=random_seed)

cv_scores = cross_val_score(final_rf,
                           X_train_selected,
                           y_train,
                           cv=cv,
                           scoring="f1_weighted",
                           n_jobs=-n_jobs,
                           verbose=1)

print(f"Cross-validation F1-weighted: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

# Train the final model on all training data
final_rf.fit(X_train_selected, y_train)

# Save the final model
joblib.dump(final_rf, model_path)

Final model parameters: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 5, 'max_features': 'sqrt', 'max_depth': 50, 'class_weight': 'balanced'}


[Parallel(n_jobs=-10)]: Using backend LokyBackend with 5 concurrent workers.
[Parallel(n_jobs=-10)]: Done  30 out of  30 | elapsed:  5.8min finished


Cross-validation F1-weighted: 0.909 ± 0.007


['/home/mariya/Data/pallas/training/gex/gex.model.pkl']

In [14]:
# ----------------------
# Evaluate final model on test set
# ----------------------

# Use selected features for test set
y_pred_final = final_rf.predict(X_test_selected)

# Print size of selected features
print(f"Number of selected features: {len(selected_features)}")

# Print shape of X_test_selected
print(f"Shape of X_test_selected: {X_test_selected.shape}")

print("\nTest set classification report (final optimized model):")
print(classification_report(y_test, y_pred_final, digits=3))


Number of selected features: 3920
Shape of X_test_selected: (770, 3920)

Test set classification report (final optimized model):
                          precision    recall  f1-score   support

      AML, APL-PML::RARA      1.000     1.000     1.000         4
          AML, BCR::ABL1      0.000     0.000     0.000         1
     AML, CBFA2T3::GLIS2      1.000     0.714     0.833         7
        AML, CBFB::MYH11      1.000     1.000     1.000        34
           AML, CEBPAalt      1.000     0.929     0.963        14
        AML, DEK::NUP214      1.000     1.000     1.000         7
             AML, ETV6-r      1.000     0.500     0.667         2
           AML, FUS::ERG      1.000     1.000     1.000         1
            AML, KAT6A-r      1.000     1.000     1.000         2
            AML, KMT2A-r      0.960     0.960     0.960        25
            AML, MECOM-r      0.500     1.000     0.667         1
            AML, NPM1alt      0.879     1.000     0.935        29
            

/home/mariya/miniconda3/envs/pallas/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/mariya/miniconda3/envs/pallas/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/mariya/miniconda3/envs/pallas/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

In [18]:

# ----------------------
# Extract top genes from final model
# ----------------------
# Get feature importances from final RF
importances = final_rf.feature_importances_

# Rank genes by importance
gene_importances = pd.DataFrame({
    "Gene": selected_features,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

gt = GeneThesaurus()
translated = gt.translate_genes(gene_importances["Gene"].tolist(),
                               source="ensembl_id", target="symbol")
gene_importances["Gene"] = gene_importances["Gene"].map(translated)

print("\nTop 20 genes driving classification (final model):")
print(gene_importances.head(20))


Top 20 genes driving classification (final model):
         Gene  Importance
3082    NUTM1    0.003348
2068    AGAP1    0.003289
1768   MEGF10    0.002871
330   RUNX1T1    0.002598
3616    APELA    0.002317
515     SALL4    0.002232
2109  IGF2BP1    0.002213
327     HOXA9    0.002196
2325     UNCX    0.002156
137     HDAC9    0.002124
3630   HOXA10    0.002112
650     KCNN1    0.002037
3198   SOWAHD    0.002033
1194    KRT17    0.002019
3716    ANXA8    0.002015
741     HOXB6    0.002010
371     MECOM    0.001917
3097   ANKS1B    0.001909
1893   VPS26B    0.001848
3125     PBX1    0.001824


In [19]:
# How many of the feature importances are zero?
num_zero_importances = np.sum(importances == 0)
print(f"\nNumber of features with zero importance: {num_zero_importances} out of {len(importances)}")

# Print lowest importance genes as well
print("\nBottom 20 genes by importance (final model):")
print(gene_importances.tail(20))


Number of features with zero importance: 0 out of 3920

Bottom 20 genes by importance (final model):
              Gene    Importance
2158       CYP11B1  6.281153e-06
3740           NaN  5.846135e-06
3684         HSPB9  5.811843e-06
3072          SIX6  5.806994e-06
447            MLN  5.782854e-06
3855  SENP3-EIF4A1  5.403627e-06
977           GHRH  5.371631e-06
1141         GFRA4  5.139705e-06
3765      C17orf50  4.765897e-06
3729           NaN  4.738409e-06
3615           NaN  4.485539e-06
3776           NaN  3.782512e-06
3811        NUDT18  3.597458e-06
1751         CCNA2  3.409343e-06
3757           NaN  2.722486e-06
3899     EEF1AKMT4  2.426592e-06
3194          SSX7  2.237563e-06
3666           NaN  2.082904e-06
3670    BUB1B-PAK6  1.320176e-06
3562       CYP21A2  5.349689e-08
